In [1]:
import gymnasium as gym
from gymnasium.spaces import Discrete

import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import numpy as np
from tqdm.auto import tqdm

from dataclasses import dataclass

In [2]:
type State = int
type Action = int
type Reward = float

In [ ]:
@dataclass
class Parameters:
  alpha: float  # step size
  epsilon: float  # e-greedy
  epsilon_decay: float
  epsilon_end: float
  gamma: float  # discount factor


def select_action(q_table: np.ndarray, state: State, epsilon: float, n_actions: int, env: gym.Env) -> Action:
  # using e-greedy method

  q: np.ndarray = q_table[state]
  assert q.shape == (n_actions,)

  # exploratory
  if env.np_random.random() < epsilon:
    a = env.action_space.sample()
  else:
    a = int(env.np_random.choice(np.flatnonzero(q == q.max())))  # break ties randomly

  return a

In [4]:
def update(
  q_table: np.ndarray,
  state: State,
  new_state: State,
  action: Action,
  reward: Reward,
  done: bool,
  params: Parameters,
):
  q_next = (1 - done) * np.max(q_table[new_state])
  td_error = reward + params.gamma * q_next - q_table[state, action]
  q_table[state, action] += params.alpha * td_error

In [5]:
def to_coord(pos: int) -> tuple[int, int]:
  x = pos % 12
  y = pos // 12

  return x, y

In [6]:
environ: gym.Env[int, int] = gym.make("CliffWalking-v1", max_episode_steps=10_000, render_mode="ansi")

In [7]:
assert isinstance(environ.action_space, Discrete)
assert isinstance(environ.observation_space, Discrete)


n_actions = int(environ.action_space.n)
obs_dim = int(environ.observation_space.n)

In [ ]:
# init params
params = Parameters(alpha=0.1, epsilon=0.1, epsilon_decay=1, epsilon_end=0.01, gamma=0.99)
n_episodes = 10000

In [ ]:
total_rewards = []

q_table = np.zeros((obs_dim, n_actions))

pbar_ep = tqdm(range(n_episodes), desc="Episodes")
for ep in pbar_ep:
  done = False

  total_reward = 0
  observation, info = environ.reset()

  while not done:
    action = select_action(q_table, observation, params.epsilon, n_actions, environ)

    # take the action
    new_observation, reward, terminated, truncated, info = environ.step(action)

    done = terminated or truncated

    # update
    update(q_table, observation, new_observation, action, float(reward), done, params)

    observation = new_observation

    params.epsilon = max(params.epsilon_end, params.epsilon*params.epsilon_decay)

    total_reward += float(reward)

  total_rewards.append(total_reward)

  print(f"Episode finished! Total reward: {total_reward}")

Episodes:   0%|          | 0/1000 [00:00<?, ?it/s]

Episode finished! Total reward: -1467.0
Episode finished! Total reward: -1178.0
Episode finished! Total reward: -382.0
Episode finished! Total reward: -809.0
Episode finished! Total reward: -465.0
Episode finished! Total reward: -250.0
Episode finished! Total reward: -129.0
Episode finished! Total reward: -357.0
Episode finished! Total reward: -377.0
Episode finished! Total reward: -188.0
Episode finished! Total reward: -76.0
Episode finished! Total reward: -96.0
Episode finished! Total reward: -475.0
Episode finished! Total reward: -72.0
Episode finished! Total reward: -46.0
Episode finished! Total reward: -273.0
Episode finished! Total reward: -262.0
Episode finished! Total reward: -84.0
Episode finished! Total reward: -76.0
Episode finished! Total reward: -50.0
Episode finished! Total reward: -63.0
Episode finished! Total reward: -107.0
Episode finished! Total reward: -255.0
Episode finished! Total reward: -39.0
Episode finished! Total reward: -58.0
Episode finished! Total reward: -

In [10]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
  y=total_rewards,
  mode='lines',
  name='Raw',
  line=dict(color='steelblue')
))

fig.update_layout(
  title='Training Rewards',
  xaxis_title='Episode',
  yaxis_title='Total Reward',
)

fig.show()

In [ ]:
# eval run

observation, info = environ.reset()

done = False

rewards = 0

while not done:
  print(environ.render())
  
  # choose an action
  action = select_action(q_table, observation, params.epsilon, n_actions, environ)

  # take the action
  new_observation, reward, terminated, truncated, info = environ.step(action)
  observation = new_observation

  rewards += float(reward)
  done = terminated or truncated

print(f"Total rewards: {rewards}")

o  o  o  o  o  o  o  o  o  o  o  o
o  o  o  o  o  o  o  o  o  o  o  o
o  o  o  o  o  o  o  o  o  o  o  o
x  C  C  C  C  C  C  C  C  C  C  T


o  o  o  o  o  o  o  o  o  o  o  o
o  o  o  o  o  o  o  o  o  o  o  o
x  o  o  o  o  o  o  o  o  o  o  o
o  C  C  C  C  C  C  C  C  C  C  T


o  o  o  o  o  o  o  o  o  o  o  o
o  o  o  o  o  o  o  o  o  o  o  o
o  x  o  o  o  o  o  o  o  o  o  o
o  C  C  C  C  C  C  C  C  C  C  T


o  o  o  o  o  o  o  o  o  o  o  o
o  o  o  o  o  o  o  o  o  o  o  o
x  o  o  o  o  o  o  o  o  o  o  o
o  C  C  C  C  C  C  C  C  C  C  T


o  o  o  o  o  o  o  o  o  o  o  o
o  o  o  o  o  o  o  o  o  o  o  o
o  x  o  o  o  o  o  o  o  o  o  o
o  C  C  C  C  C  C  C  C  C  C  T


o  o  o  o  o  o  o  o  o  o  o  o
o  o  o  o  o  o  o  o  o  o  o  o
x  o  o  o  o  o  o  o  o  o  o  o
o  C  C  C  C  C  C  C  C  C  C  T


o  o  o  o  o  o  o  o  o  o  o  o
o  o  o  o  o  o  o  o  o  o  o  o
o  x  o  o  o  o  o  o  o  o  o  o
o  C  C  C  C  C  C  C  C  C  C  T


o  o  

In [12]:
environ.close()